# Import

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
from mapd.sinq_builders import build_composite_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

%load_ext autoreload
%autoreload 2

import mapd.sentinels  # load once
%aimport -mapd.sentinels

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Load Figure 3 sinq with norpA flies

In [ ]:
plt.close('all')
fig_dir = './FigureS1_norpA'

In [ ]:
sinq = Sinq(sinqname='Fig1_control')
sinq.display_df(show_tags=True)
df_norpa = sinq.display_df(show_tags=True).loc[sinq.has_tag('norpa')].copy()
df_norpa_na = sinq.display_df(show_tags=True).loc[sinq.rows_with_tags(['norpa', 'no_antenna'],mode='all')]
df_norpa_only = df_norpa.loc[[i for i in df_norpa.index if i not in df_norpa_na.index]]

df_wt_na = sinq.display_df(show_tags=True).loc[sinq.rows_with_tags(['no_antenna'],mode='all')]

geno_map = {'Hot-Cell-Gal4 (test)':             'HC',
 'Hot-Cell-LexA_Chr;81A06_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;35c09_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;35C09_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;78E05_pJFRC7': 'HC',
 'Hot-Cell-LexA_Chr;31H05_pJFRC7': 'HC',
 'SS61350_pJFRC7':                  '+',
 '+;31H05-Gal4_pJFRC7':             '+',
 '+;TH-Gal4_UAS-Kir2_1':            '+;TH-Gal4_UAS-Kir2_1',
 '+;pJFRC7;31H05-Gal4':             '+',
 '+;pJFRC7;SS61350':                '+',
 'ppk-Gal4;10XUAS-ChR in WT':       '+;ppk-GAL4;10XUAS-ChR',
 '+;ppk-GAL4;10XUAS-ChR':           '+;ppk-GAL4;10XUAS-ChR',
 'Hot-Cell-Gal4_ChrinWT':           'HC',
 'w, NorpA':                        'NorpA',
 '+;pJFRC7;+':                      '+',
 'norpA;cry':                       '+',
 '+;pJFRC7;35C09-GAL4':             '+',
 'ss61350_pJFRC7':                  '+',
 'norpAE55':                        'NorpA',
 '+;pFJRC7;+':                      '+',
 'norpA':                           'NorpA',
 '+;pJFRC7;81A06-GAL4':             '+',
}

df_wt_na['geno_smpl'] = df_wt_na['genotype'].map(geno_map)
df_wt_na = df_wt_na.loc[df_wt_na['geno_smpl']=='+']


In [ ]:
sinq2 = Sinq(sinqname='Figure1')
df_wt_only = sinq2.display_df(show_tags=True).loc[sinq2.rows_with_tags(['learn'],mode='all')]
df_wt_only['geno_smpl'] = df_wt_only['genotype'].map(geno_map)
df_wt_only = df_wt_only.loc[df_wt_only['geno_smpl']=='+']
df_wt_only


In [ ]:
df_wt_replay = sinq.display_df(show_tags=True).loc[sinq.rows_with_tags(['replay'],mode='all')]
# df_wt_replay['geno_smpl'] = df_wt_replay['genotype'].map(geno_map)
# df_wt_replay = df_wt_replay.df_wt_replay[df_wt_only['geno_smpl']=='+']
df_wt_replay

# Create file paths

In [ ]:
def mk_dir(id):
    folder_path = f'./{fig_dir}/{id}'
    os.makedirs(folder_path, exist_ok=True)
    export_path = f'{folder_path}/exports'
    os.makedirs(export_path, exist_ok=True)

for id in df_wt_na.index:
    mk_dir(id)

for id in df_norpa_only.index:
    mk_dir(id)

for id in df_norpa_na.index:
    mk_dir(id)

for id in df_wt_replay.index:
    mk_dir(id)

for id in df_wt_only.index:
    mk_dir(id)

    # T = sinq2.restore_table(id)
    # T.extract_trial_properties()
    # fig,ax = T.plot_outcomes(savefig=True,format='png',fig_dir=f'./Figure2/{id}')

    # T.compute_trial_method('prestim_holding_cost')
    # fig,ax = T.plot_trial_computations('prestim_holding_cost',savefig=True,fig_dir=f'./Figure2/{id}')

    # T.compute_trial_method('prestim_v_rms')
    # fig,ax = T.plot_trial_computations('prestim_v_rms',savefig=True,fig_dir=f'./Figure2/{id}')

    # T.compute_trial_method('probe_effort')
    # fig,ax = T.plot_trial_computations('probe_effort',savefig=True,fig_dir=f'./Figure2/{id}')

    # T.compute_trial_method('probe_positive_effort')
    # fig,ax = T.plot_trial_computations('probe_positive_effort',savefig=True,fig_dir=f'./Figure2/{id}')

    # T.compute_trial_method('probe_rms_velocity')
    # fig,ax = T.plot_trial_computations('probe_rms_velocity',savefig=True,fig_dir=f'./Figure2/{id}')
    # display(fig)


plt.close('all')

# How much cummulative punishment do flies receive?

In [ ]:
import matplotlib.patches as patches
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

def cumulative_punishment(T:Table = None,cpdf = None, fig = None, ax = None, dfc = None, index = None, fig_dir:str = fig_dir,dur_v_trial_flag = False,color=None):
    output_directory = None

    if not T is None:
        fig = Figure(figsize=(6, 6), dpi=200)
        canvas = FigureCanvas(fig)
        ax = fig.add_subplot(111)
        if index is None:
            index = T.df.index
            reindex = False
        else:
            reindex = True
        # cpdf = T.df.loc[index,['pyasState','pyasXPosition','pyasWidth','probeZero','as_duration','on_target','time_on_target','op_cnd_blocks','total_duration']].copy()
        cpdf = T.df.loc[index,['pyasState','pyasXPosition','pyasWidth','probeZero','as_duration','op_cnd_blocks','total_duration']].copy()
        if reindex:
            cpdf = cpdf.reset_index(drop=True)
        cpdf['cum_pun'] = cpdf['as_duration'].cumsum()
        cpdf['cum_dur'] = cpdf['total_duration'].cumsum()
        dfc = T.dfc
        output_directory = f'{fig_dir}/{T.dfc}'
        export_path = f'{fig_dir}/{T.dfc}/exports'
        

    rocket_cmap = sns.color_palette("rocket", as_cmap=True)

    cmin = 10
    cmax = -500

    _force_clrs = [
        (np.float64(0.95447591), np.float64(0.47082238), np.float64(0.32310953)),
        (np.float64(0.7965014), np.float64(0.10506637), np.float64(0.31063031)),
        ]
    if not dur_v_trial_flag:
        for ocb in cpdf['op_cnd_blocks'].unique():
            # print(ocb)
            T_rows = cpdf[cpdf['op_cnd_blocks']==ocb]
            
            tr_min = T_rows.index.min()
            tr_max = T_rows.index.max()
            width = tr_max-tr_min
            cum_block_min = T_rows.loc[tr_min,'cum_pun']
            cum_block_max = T_rows.loc[tr_max,'cum_pun']
            height = cum_block_max-cum_block_min

            # target patches
            tgt_clr = _force_clrs[0] if T_rows.loc[tr_min,'pyasState']=='lo' else _force_clrs[1]
            rect = patches.Rectangle(
                    (tr_min,cum_block_min),        # Bottom-left corner of the rectangle
                    width,           # Width (covers the specified rows)
                    height, # Height (covers all categories)    
                    linewidth=0,                        
                    edgecolor='none',
                    facecolor=tgt_clr,
                    alpha=.2
                )
            ax.add_patch(rect)

    y = cpdf['cum_pun']
    if dur_v_trial_flag:
        x = cpdf['cum_dur']/60
        x_label = 'Experiment duration (min)'
    else:
        x = cpdf.index
        x_label = 'Trial number'

    if (not T is None) or (not color is None):
        ax.plot(x,y,color=color,label=dfc)
    else:
        ax.plot(x,y,label=dfc)

    # ax.plot(T.df.index,T.df['on_target'].cumsum())
    # ax.plot(T.df.index,T.df['time_on_target'].cumsum())
    ax.set_xlabel(x_label)
    ax.set_ylabel("Cumulative punishment (s)")
    ax.set_title(f'Cumulative punishment')
    
    if not output_directory is None:    
        # fig.savefig(f'{output_directory}/cumulative_punishment{T.dfc}.svg',format='svg')
        fig.savefig(f'{output_directory}/cumulative_punishment{T.dfc}.png',format='png')
        cpdf.to_pickle(path=f'{export_path}/cumulative_punishment_{T.dfc}.pkl')

    return fig,ax



In [ ]:
def run_pun(dfc):
    T = sinq.restore_table(dayflycell=dfc)
    T.extract_trial_properties()
    # T.extract_trial_properties(prop_list = ['on_target'])
    # T.extract_trial_properties(prop_list = ['time_on_target'])
    as_off_idx = T.df.loc[T.df['as_outcome'].isin(['as_off','timeout','as_off_late','timeout_fail'])].index
    sinq2.drop_tables()
    return cumulative_punishment(T,index=as_off_idx)
    
    # sinq2.drop_tables()

# for id in df_wt_na.index:
#     run_pun(id)

# for id in df_norpa_only.index:
#     run_pun(id)

# for id in df_norpa_na.index:
#     run_pun(id)

# for id in df_wt_replay.index:
#     run_pun(id)


def run_pun_2(dfc):
    T = sinq2.restore_table(dayflycell=dfc)
    T.extract_trial_properties()
    # T.extract_trial_properties(prop_list = ['on_target'])
    # T.extract_trial_properties(prop_list = ['time_on_target'])
    as_off_idx = T.df.loc[T.df['as_outcome'].isin(['as_off','timeout','as_off_late','timeout_fail'])].index
    sinq2.drop_tables()
    return cumulative_punishment(T,index=as_off_idx)
    

for id in df_wt_only.index:
    run_pun_2(id)



# Reload exports and plot together

In [ ]:
# Now load and plot the plots together
from collections import defaultdict
cpdf = defaultdict(dict)

ftrfig = Figure(figsize=(12,12))
# plt.figure(figsize=(5,5))
ftrax = ftrfig.add_subplot(1, 1, 1)
dur_v_trial_flag = False

def add_condition_to_cumpun(dfc,ftrfig,ftrax,color='#000000'):
    export_path = f'./{fig_dir}/{dfc}/exports'
    pklpat = f'{export_path}/cumulative_punishment_{dfc}.pkl'
    file_path = glob.glob(pklpat)
    if file_path:
        file_path = file_path[0]
        print(file_path)

    cpdf[dfc] = pd.read_pickle(file_path)
    # cpdf[dfc]['cum_dur'] = cpdf[dfc]['total_duration'].cumsum()
    ftrfig,ftrax = cumulative_punishment(cpdf=cpdf[dfc],fig = ftrfig,ax=ftrax,dur_v_trial_flag=dur_v_trial_flag,dfc=dfc,color=color)
    return ftrfig,ftrax

colors = {
    'wt_na':          '#000000',
    'norpa_only':     '#FF0000',
    'norpa_na':         "#FF00FF",
    'wt_only':          '#00FF00'
}

for dfc in df_wt_na.index:
    ftrfig,ftrax = add_condition_to_cumpun(dfc=dfc,ftrfig=ftrfig,ftrax=ftrax,color=colors['wt_na'])

for dfc in df_norpa_only.index:
    ftrfig,ftrax = add_condition_to_cumpun(dfc=dfc,ftrfig=ftrfig,ftrax=ftrax,color=colors['norpa_only'])

for dfc in df_norpa_na.index:
    ftrfig,ftrax = add_condition_to_cumpun(dfc=dfc,ftrfig=ftrfig,ftrax=ftrax,color=colors['norpa_na'])

for dfc in df_wt_only.index:
    ftrfig,ftrax = add_condition_to_cumpun(dfc=dfc,ftrfig=ftrfig,ftrax=ftrax,color=colors['wt_only'])

if not dur_v_trial_flag:
    x = [0,  400]
    y = [0, 400*10]
    ftrax.plot(x,y,ls='--',color='gray')

    x = [0,  1000]
    y = [0, 1000*1]
    ftrax.plot(x,y,ls='--',color='gray')

ftrax.legend()
display(ftrfig)
ftrfig.savefig(f'./{fig_dir}/cumulative_punishment_over_trials_all_flies_trials.svg')


# Average no_as trials, timeout trials, etc.

In [ ]:
sinq.df.index

In [ ]:
def plot_average_as_off(T,outcome_list = ['as_off','timeout','as_off_late','timeout_fail'],n_pre  = 100,n_post = 800):
    as_off_idx = T.df.loc[T.df['as_outcome'].isin(outcome_list)].index

    trials = T.df.loc[as_off_idx,'Trial']

    def closest_zero_idx(x):
        return int(np.nanargmin(np.abs(x)))

    def dpp_snippet(tr):
        dpp = tr.probe_position[tr.downsample_probe]-tr.probeZero
        x = tr.time[tr.downsample_probe]

        i0 = closest_zero_idx(x)
        x = x-x[i0]

        if i0 < n_pre or len(x) - i0 - 1 < n_post:
            raise ValueError("Not enough data around zero")
            
        x_win   = x[i0-n_pre : i0+n_post+1]
        dpp_win = dpp[i0-n_pre : i0+n_post+1]

        return pd.Series(
            {"x": x_win, "dpp": dpp_win}
        )
        # return x_win,dpp_win


    out = trials.apply(dpp_snippet)

    X = np.stack(out['x'].values)     # shape: (n_trials, 901)
    DPP = np.stack(out['dpp'].values) # shape: (n_trials, 901)

    x_mean   = np.nanmean(X, axis=0)
    dpp_mean = np.nanmean(DPP, axis=0)
    dpp_mean = np.squeeze(dpp_mean, axis=-1)
    dpp_std = np.nanstd(DPP, axis=0,ddof=1)

    n_trials = np.sum(~np.isnan(DPP), axis=0)
    dpp_sem = dpp_std / np.sqrt(n_trials)

    dpp_std = np.squeeze(dpp_std, axis=-1)
    dpp_sem = np.squeeze(dpp_sem, axis=-1)
    
    l = n_pre+n_post+1
    assert x_mean.shape == (l,)
    assert dpp_mean.shape == (l,)
    assert dpp_std.shape == (l,)
    assert dpp_sem.shape == (l,)


    df_summary = pd.DataFrame({
        "t": x_mean,        # shape (901,)
        "dpp_mean": dpp_mean,
        "dpp_std": dpp_std,
    })

    fig = Figure(figsize=(2, 2), dpi=200)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111)

    ax.fill_between(
        x_mean,
        dpp_mean - dpp_std,
        dpp_mean + dpp_std,
        color="lightgray",
        alpha=0.4,
        linewidth=0,
        label="± STD",
    )

    ax.fill_between(
        x_mean,
        dpp_mean - dpp_sem,
        dpp_mean + dpp_sem,
        color="#8ecae6",   # light blue
        alpha=0.6,
        linewidth=0,
        label="± SEM",
    )

    ax.plot(
        x_mean,
        dpp_mean,
        color="black",
        linewidth=2,
        label="Mean",
    )

    ax.axvline(0, color="k", linestyle="--")
    ax.set_title(f'{T.dfc}: [{'_'.join(outcome_list)}]')
    ax.set_xlabel('time (s)')
    ax.set_ylabel('probe position (um)')

    export_path = f'{fig_dir}/{T.dfc}/exports'
    dfc_fig_dir = f'{fig_dir}/{T.dfc}'   

    canvas.print_png(f'{dfc_fig_dir}/average_response_{T.dfc}_{'_'.join(outcome_list)}.png')
    df_summary.to_pickle(path=f'{export_path}/as_off_average_{T.dfc}_{'_'.join(outcome_list)}.pkl')
    display(fig)



In [ ]:
# idx = sinq.has_tag(tag='replay')
# sinq.drop_tables(index=idx)
# sinq.display_df()

idx = sinq.df.index[sinq.has_tag(tag='no_antenna')]
for dfc in idx:
    sinq.restore_table(dayflycell=dfc)

In [ ]:
df_wt_na
df_norpa_na
df_norpa_only

In [ ]:
for dfc in df_wt_na.index:
    T = sinq.restore_table(dayflycell=dfc)
    plot_average_as_off(T,n_post=1200)
    plot_average_as_off(T,outcome_list=['as_off'],n_post=1200)  

for dfc in df_norpa_na.index:
    T = sinq.restore_table(dayflycell=dfc)
    plot_average_as_off(T,n_post=1200)
    plot_average_as_off(T,outcome_list=['as_off'],n_post=1200)  

for dfc in df_norpa_only.index:
    T = sinq.restore_table(dayflycell=dfc)
    plot_average_as_off(T,n_post=1200)
    plot_average_as_off(T,outcome_list=['as_off'],n_post=1200)  


In [ ]:

for dfc in df_wt_only.index:
    T = sinq2.restore_table(dayflycell=dfc)
    plot_average_as_off(T,n_post=1200)
    plot_average_as_off(T,outcome_list=['as_off'],n_post=1200)  
    sinq2.drop_tables()

# Plot averages for each

In [ ]:
outcome_list=['as_off']

mean_df_dict = {
    'wt_only': None,
    'wt_na': None,
    'norpa_na': None,
    'norpa_only': None,
}
std_df_dict = {
    'wt_only': None,
    'wt_na': None,
    'norpa_na': None,
    'norpa_only': None,
}

for df,nm in zip([df_wt_only,df_wt_na,df_norpa_na,df_norpa_only],mean_df_dict.keys()):
    mean_cols = {}
    std_cols = {}
    t_ref = None
    for dfc in df.index:
        # df_summary.to_pickle(path=f'{export_path}/as_off_average_{dfc}_{'_'.join(outcome_list)}.pkl')  
        export_path = f'{fig_dir}/{dfc}/exports'
        df_summary = pd.read_pickle(f'{export_path}/as_off_average_{dfc}_{'_'.join(outcome_list)}.pkl')

        if t_ref is None:
            t_ref = df_summary["t"].values
        else:
            # assert np.allclose(t_ref, df_summary["t"].values,rtol=1e-3)
            ...

        mean_cols[dfc] = df_summary["dpp_mean"].values
        std_cols[dfc]  = df_summary["dpp_std"].values

    mean_df_dict[nm] = pd.DataFrame(mean_cols, index=t_ref)
    std_df_dict[nm]  = pd.DataFrame(std_cols,  index=t_ref)


In [ ]:
fig = Figure(figsize=(6, 6), dpi=200)
canvas = FigureCanvas(fig)
ax = fig.add_subplot(111)

colors = {
    "wt_na": "black",
    "norpa_na": "#e63946",     # red
    "norpa_only": "#457b9d",   # blue
}

for cond, mean_df in mean_df_dict.items():
    t = mean_df.index.values               # (901,)
    Y = mean_df.values                     # (901, N_flies)

    mean = np.nanmean(Y, axis=1)
    std  = np.nanstd(Y, axis=1, ddof=1)
    n    = np.sum(~np.isnan(Y), axis=1)
    sem  = std / np.sqrt(n)

    color = colors.get(cond, "gray")

    # SEM band
    ax.fill_between(
        t,
        mean - sem,
        mean + sem,
        color=color,
        alpha=0.25,
        linewidth=0,
    )

    # Mean trace
    ax.plot(
        t,
        mean,
        color=color,
        linewidth=2,
        label=cond,
    )

    for dfc in mean_df.columns:
        # individual traces
        ax.plot(
            t,
            mean_df[dfc],
            color=color,
            linewidth=.5,
            label=dfc,
        )

ax.axvline(0, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("dpp")
ax.legend(frameon=False)

# plt.tight_layout()
display(fig)

In [ ]:
fig.savefig(fname=f'{fig_dir}/as_off_responses.svg',format='svg')

# Plot factors to support norpA/antennal importance

In [ ]:
for df in [df_wt_na,df_norpa_na,df_norpa_only,df_wt_only]:
    df['rms_velocity_bar'] = df['rms_velocity'] / df['duration']
    df['holding_cost_bar'] = df['holding_cost'] / df['duration']
    df['probe_positive_effort_bar'] = df['probe_positive_effort'] / df['duration']
    print(df['holding_cost_bar'])

In [ ]:
for df in [df_wt_na,df_norpa_na,df_norpa_only]:
    print(df['holding_cost_bar'])

In [ ]:
import numpy as np
import plotly.graph_objects as go

def plot_factor_across_conditions(
    dfs: dict[str, pd.DataFrame],
    factor_col: str = "holding_cost_bar",
    colors: dict[str, str] | None = None,
    jitter: float = 0.18,
    seed: int = 0,
    title: str | None = None,
):
    rng = np.random.RandomState(seed)

    if colors is None:
        colors = {}

    fig = go.Figure()
    order = list(dfs.keys())

    for xi, cond in enumerate(order):
        df = dfs[cond].dropna(subset=[factor_col])

        y = df[factor_col].values
        x = xi + rng.uniform(-jitter, jitter, size=len(y))

        color = colors.get(cond, "gray")

        # ---- scatter (individual flies) ----
        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode="markers",
                name=cond,
                legendgroup=cond,
                marker=dict(
                    size=8,
                    color=color,
                    opacity=0.7,
                    line=dict(width=0),
                ),
                customdata=df.index.astype(str),
                hovertemplate=(
                    f"{cond}"
                    "<br>dfc: %{customdata}"
                    f"<br>{factor_col}: %{{y:.4g}}"
                    "<extra></extra>"
                ),
                showlegend=True,
            )
        )

        # ---- bar (mean ± SEM) ----
        mean = np.mean(y)
        sem = np.std(y, ddof=1) / np.sqrt(len(y)) if len(y) > 1 else 0

        fig.add_trace(
            go.Bar(
                x=[xi],
                y=[mean],
                width=0.35,
                name=f"{cond} mean",
                legendgroup=cond,
                marker=dict(color=color, opacity=0.45),
                error_y=dict(
                    type="data",
                    array=[sem],
                    visible=True,
                    thickness=1.5,
                    width=6,
                ),
                showlegend=False,
            )
        )

    # ---- axes + layout ----
    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(order))),
        ticktext=order,
    )

    fig.update_yaxes(title_text=factor_col)

    if title is None:
        title = f"{factor_col} across conditions"

    fig.update_layout(
        title=title,
        width=700,
        height=500,
        margin=dict(l=60, r=40, t=60, b=50),
        hoverlabel=dict(font_size=14),
    )

    return fig

In [ ]:
dfs = {
    "wt_only": df_wt_only,
    "wt_na": df_wt_na,
    "norpa_only": df_norpa_only,
    "norpa_na": df_norpa_na,
}

factor_col = "holding_cost_bar"

colors = {
    "wt_na": "black",
    "norpa_na": "#e63946",      # red
    "norpa_only": "#457b9d",    # blue
    "wt_only": "lightgray",
}

In [ ]:
factor_col = 'rms_velocity_bar'
fig = plot_factor_across_conditions(
    dfs=dfs,
    factor_col=factor_col,
    colors=colors,
)

fig.show()
fig.write_image(f'{fig_dir}/{factor_col}_wt_na_norpa_norpana_bar.svg','svg')
fig.write_html(f'{fig_dir}/{factor_col}_wt_na_norpa_norpana_bar.html')